# Tool Calling & ReAct Agent — AI Engineering Interview Prep

Claude tool use API, function calling, and building a ReAct agent loop.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os
import json
import math
import re
from datetime import datetime

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"

print(f"Model: {MODEL}")

## 1. Define Tools

In [ ]:
# Tool definitions (JSON schema format for Anthropic API)
TOOLS = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use for arithmetic, statistics, and numerical computations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Python math expression to evaluate, e.g. 'math.sqrt(144)' or '(0.95 ** 10) * 100'"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_ml_knowledge",
        "description": "Search a local ML knowledge base for definitions, formulas, and comparisons.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query about ML concepts"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_model_benchmark",
        "description": "Retrieve benchmark scores for common ML models on standard datasets.",
        "input_schema": {
            "type": "object",
            "properties": {
                "model_name": {"type": "string", "description": "Name of the ML model"},
                "dataset": {"type": "string", "description": "Dataset name (optional)", "default": ""}
            },
            "required": ["model_name"]
        }
    }
]

# Tool implementations
KNOWLEDGE_BASE = {
    "precision": "Precision = TP / (TP + FP). Fraction of positive predictions that are correct.",
    "recall": "Recall = TP / (TP + FN). Fraction of actual positives correctly identified.",
    "f1": "F1 = 2 * (Precision * Recall) / (Precision + Recall). Harmonic mean of precision and recall.",
    "attention": "Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V. Computes weighted sum of values.",
    "lora": "LoRA freezes pretrained weights and adds low-rank matrices: W_new = W + alpha/r * B*A.",
}

BENCHMARKS = {
    "bert": {"GLUE": 79.6, "SQuAD 1.1": 88.5},
    "gpt-4": {"MMLU": 86.4, "HumanEval": 67.0},
    "resnet-50": {"ImageNet Top-1": 76.1, "ImageNet Top-5": 92.9},
    "random forest": {"UCI Adult": 86.2, "MNIST": 97.1},
    "xgboost": {"Higgs": 83.0, "KDD Cup 99": 99.2},
}

def execute_tool(tool_name, tool_input):
    if tool_name == "calculate":
        try:
            result = eval(tool_input["expression"], {"math": math, "__builtins__": {}})
            return {"result": round(float(result), 6)}
        except Exception as e:
            return {"error": str(e)}

    elif tool_name == "search_ml_knowledge":
        q = tool_input["query"].lower()
        for key, value in KNOWLEDGE_BASE.items():
            if key in q:
                return {"result": value}
        return {"result": "Not found in knowledge base."}

    elif tool_name == "get_model_benchmark":
        model = tool_input["model_name"].lower()
        dataset = tool_input.get("dataset", "")
        if model in BENCHMARKS:
            scores = BENCHMARKS[model]
            if dataset:
                return {"result": scores.get(dataset, f"No benchmark for {dataset}")}
            return {"result": scores}
        return {"result": f"No benchmark data for {model}"}

    return {"error": f"Unknown tool: {tool_name}"}

print(f"{len(TOOLS)} tools defined")

## 2. Single Tool Call

In [ ]:
# Simple tool use: ask Claude to calculate something
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=TOOLS,
    messages=[{"role": "user", "content": "If a model has 95% precision and 80% recall, what is its F1 score? Calculate it."}]
)

print(f"Stop reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"Tool called: {block.name}")
        print(f"Input: {block.input}")

        # Execute tool
        tool_result = execute_tool(block.name, block.input)
        print(f"Result: {tool_result}")

        # Return result to Claude
        final = client.messages.create(
            model=MODEL, max_tokens=200, tools=TOOLS,
            messages=[
                {"role": "user", "content": "If a model has 95% precision and 80% recall, what is its F1 score? Calculate it."},
                {"role": "assistant", "content": response.content},
                {"role": "user", "content": [{"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(tool_result)}]}
            ]
        )
        print(f"\nFinal answer: {final.content[0].text}")
    elif block.type == "text":
        print(f"Text: {block.text}")

## 3. ReAct Agent Loop

In [ ]:
def run_agent(user_query, max_iterations=5):
    """ReAct agent: Reason → Act (tool call) → Observe → repeat until done."""
    messages = [{"role": "user", "content": user_query}]
    print(f"Query: {user_query}")
    print("=" * 60)

    for iteration in range(max_iterations):
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            tools=TOOLS,
            system="You are an ML research assistant. Use tools to compute and look up information before answering. Always use tools when calculations or lookups are needed.",
            messages=messages
        )

        # Collect tool calls and text from this response
        tool_calls = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]

        if text_blocks:
            for t in text_blocks:
                if t.text.strip():
                    print(f"[Thought] {t.text.strip()[:150]}")

        # Done — no more tool calls
        if response.stop_reason == "end_turn" or not tool_calls:
            final_text = next((b.text for b in response.content if b.type == "text"), "")
            print(f"\n[Final Answer]\n{final_text.strip()}")
            break

        # Execute all tool calls
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for tc in tool_calls:
            result = execute_tool(tc.name, tc.input)
            print(f"[Action] {tc.name}({tc.input}) → {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tc.id,
                "content": json.dumps(result)
            })
        messages.append({"role": "user", "content": tool_results})

# Test the agent
run_agent("Compare BERT and XGBoost: look up their benchmark scores and compute the ratio of their GLUE/UCI-Adult accuracies.")

## 4. Parallel Tool Calls

In [ ]:
# Claude can call multiple tools in a single response
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=TOOLS,
    messages=[{"role": "user", "content": "Look up what precision and recall mean, then calculate F1 if precision=0.85 and recall=0.90."}]
)

tool_calls = [b for b in response.content if b.type == "tool_use"]
print(f"Number of parallel tool calls: {len(tool_calls)}")
for tc in tool_calls:
    result = execute_tool(tc.name, tc.input)
    print(f"  {tc.name}({json.dumps(tc.input)}) → {result}")

## Interview Tips

- **Tool schema**: clear `description` + `input_schema` is critical — model uses these to decide when/how to call tools.
- **`stop_reason='tool_use'`**: Claude pauses and waits for tool results. `'end_turn'` means done.
- **Parallel tool calls**: Claude can call multiple tools in one turn — always handle multiple blocks.
- **Tool result format**: `{type: tool_result, tool_use_id: ..., content: string}` — content must be a string.
- **Max iterations**: always add a loop limit to prevent runaway agents.
- **Error handling**: return error messages from tools so Claude can recover gracefully.